# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [1]:
"""
data_inspector.py
=================
Reusable EDA toolkit for Google Colab.

Classes
-------
DataInspector   – ingestion, cleaning, normalisation, statistical analysis
PlottingMethods – granular chart generation (Bar, Pie, Histogram)

Usage (Colab)
-------------
    from data_inspector import DataInspector, PlottingMethods

    di = DataInspector()
    di.upload_data()          # triggers Colab file-upload widget
    di.data_summary()
    di.handle_missing_values(strategy="median")
    di.handle_outliers(columns=["Age", "Fare"], action="delete")
    di.plot_univariate("Age")
    di.plot_all_associations_heatmap()
"""

from __future__ import annotations

import io
import warnings
from typing import List, Optional, Literal, Union

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, LabelEncoder

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

GARBAGE_STRINGS = {"?", "n/a", "na", "null", "none", "nan", "", " ", "-", "--", "N/A",
                   "NA", "NULL", "None", "NaN", "#N/A", "#NA", "missing", "MISSING"}


def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Compute Cramér's V between two categorical series."""
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion, correction=False)[0]
    n = confusion.sum().sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2corr = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2corr / denom))


def _eta_squared(numeric: pd.Series, categorical: pd.Series) -> float:
    """Compute η (Eta) – effect size of categorical on numeric via one-way ANOVA."""
    groups = [numeric[categorical == cat].dropna().values
              for cat in categorical.dropna().unique()]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
    grand_mean = numeric.dropna().mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = sum((x - grand_mean) ** 2 for g in groups for x in g)
    return float(ss_between / ss_total) if ss_total > 0 else 0.0


# ---------------------------------------------------------------------------
# PlottingMethods
# ---------------------------------------------------------------------------

class PlottingMethods:
    """
    Granular, standalone chart generators.

    Every method returns an HTML string containing a self-contained Plotly figure
    suitable for embedding in a notebook or web page.

    Methods
    -------
    bar_chart(series, title, color)
    pie_chart(series, title)
    histogram(series, title, bins, color)
    """

    # ------------------------------------------------------------------ #
    @staticmethod
    def _to_html(fig: go.Figure) -> str:
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    # ------------------------------------------------------------------ #
    @staticmethod
    def bar_chart(
        series: pd.Series,
        title: str = "Bar Chart",
        color: str = "#636EFA",
        show_pct: bool = True,
    ) -> str:
        """
        Render a bar chart of value counts.

        Parameters
        ----------
        series : pd.Series
            Data to plot.
        title : str
            Chart title.
        color : str
            Bar fill colour (hex or CSS name).
        show_pct : bool
            If True, annotate each bar with its percentage of total.

        Returns
        -------
        str
            HTML string wrapping a Plotly figure.
        """
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        counts = series.value_counts()
        pct = (counts / counts.sum() * 100).round(1)
        text = [f"{p}%" for p in pct] if show_pct else None

        fig = go.Figure(
            go.Bar(
                x=counts.index.astype(str),
                y=counts.values,
                marker_color=color,
                text=text,
                textposition="outside",
            )
        )
        fig.update_layout(
            title=title,
            xaxis_title=series.name or "Category",
            yaxis_title="Count",
            template="plotly_white",
        )
        return PlottingMethods._to_html(fig)

    # ------------------------------------------------------------------ #
    @staticmethod
    def pie_chart(series: pd.Series, title: str = "Pie Chart") -> str:
        """
        Render a pie chart of value counts with percentage labels.

        Parameters
        ----------
        series : pd.Series
            Categorical data.
        title : str
            Chart title.

        Returns
        -------
        str
            HTML string.
        """
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        counts = series.value_counts()
        fig = go.Figure(
            go.Pie(
                labels=counts.index.astype(str),
                values=counts.values,
                textinfo="label+percent",
                hole=0.35,
            )
        )
        fig.update_layout(title=title, template="plotly_white")
        return PlottingMethods._to_html(fig)

    # ------------------------------------------------------------------ #
    @staticmethod
    def histogram(
        series: pd.Series,
        title: str = "Histogram",
        bins: int = 30,
        color: str = "#EF553B",
    ) -> str:
        """
        Render a histogram with a KDE overlay.

        Parameters
        ----------
        series : pd.Series
            Numeric data.
        title : str
            Chart title.
        bins : int
            Number of histogram bins.
        color : str
            Bar fill colour.

        Returns
        -------
        str
            HTML string.
        """
        if series is None or series.dropna().empty:
            return "<p>No data provided.</p>"

        clean = series.dropna()
        fig = go.Figure()
        fig.add_trace(
            go.Histogram(x=clean, nbinsx=bins, name="Count",
                         marker_color=color, opacity=0.75)
        )

        # KDE overlay
        kde_x = np.linspace(clean.min(), clean.max(), 300)
        kde = stats.gaussian_kde(clean)
        scale = len(clean) * (clean.max() - clean.min()) / bins
        fig.add_trace(
            go.Scatter(x=kde_x, y=kde(kde_x) * scale,
                       mode="lines", name="KDE",
                       line=dict(color="black", width=2))
        )
        fig.update_layout(
            title=title,
            xaxis_title=series.name or "Value",
            yaxis_title="Count",
            template="plotly_white",
            barmode="overlay",
        )
        return PlottingMethods._to_html(fig)


# ---------------------------------------------------------------------------
# DataInspector
# ---------------------------------------------------------------------------

class DataInspector:
    """
    End-to-end CSV data inspection, cleaning, feature engineering, and
    visualisation toolkit designed for Google Colab.

    Attributes
    ----------
    df : pd.DataFrame or None
        The active working dataset.

    Quick Start
    -----------
    >>> di = DataInspector()
    >>> di.upload_data()
    >>> di.data_summary()
    >>> di.handle_missing_values(strategy="median")
    >>> di.plot_univariate("Age")
    >>> di.plot_all_associations_heatmap()
    """

    def __init__(self) -> None:
        self.df: Optional[pd.DataFrame] = None
        self._original_df: Optional[pd.DataFrame] = None
        self._pm = PlottingMethods()

    # ================================================================== #
    # 1. DATA INGESTION & SANITISATION
    # ================================================================== #

    def upload_data(self, filepath: Optional[str] = None) -> None:
        """
        Load a CSV into ``self.df``.

        In Google Colab (no ``filepath`` supplied) this triggers the file-upload
        widget.  Outside Colab supply a local path directly.

        Parameters
        ----------
        filepath : str, optional
            Path to a CSV file.  When *None* and running in Colab, the built-in
            ``google.colab.files.upload()`` dialog is used.

        Side Effects
        ------------
        Sets ``self.df`` and ``self._original_df``.
        """
        if filepath:
            raw = pd.read_csv(filepath, na_values=list(GARBAGE_STRINGS),
                              keep_default_na=True)
        else:
            try:
                from google.colab import files  # type: ignore
                uploaded = files.upload()
                if not uploaded:
                    print("No file uploaded.")
                    return
                name = list(uploaded.keys())[0]
                raw = pd.read_csv(
                    io.BytesIO(uploaded[name]),
                    na_values=list(GARBAGE_STRINGS),
                    keep_default_na=True,
                )
            except ImportError:
                raise RuntimeError(
                    "Not running in Colab. Pass filepath= to load a CSV directly."
                )

        raw = self._sanitise(raw)
        self.df = raw
        self._original_df = raw.copy()
        print(f"✅  Loaded dataset: {self.df.shape[0]:,} rows × {self.df.shape[1]} columns")

    # ------------------------------------------------------------------ #
    def load_dataframe(self, df: pd.DataFrame) -> None:
        """
        Load an existing DataFrame directly (useful for testing).

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame to load.
        """
        self.df = self._sanitise(df.copy())
        self._original_df = self.df.copy()
        print(f"✅  DataFrame loaded: {self.df.shape[0]:,} rows × {self.df.shape[1]} columns")

    # ------------------------------------------------------------------ #
    def _sanitise(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Internal: replace garbage strings with NaN and auto-correct column types.

        Garbage values (``'?'``, ``'n/a'``, ``'NULL'`` etc.) are mapped to
        ``np.nan``.  Each column is then tested for numeric conversion; if the
        coerced result is not entirely NaN the column is stored as numeric.

        Parameters
        ----------
        df : pd.DataFrame

        Returns
        -------
        pd.DataFrame
            Cleaned copy.
        """
        # Replace garbage strings
        df = df.replace(list(GARBAGE_STRINGS), np.nan)

        # Strip whitespace from string columns
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(list(GARBAGE_STRINGS), np.nan)

        # Auto-type correction
        for col in df.select_dtypes(include="object").columns:
            coerced = pd.to_numeric(df[col], errors="coerce")

            # Convert only if most values are numeric
            if coerced.notna().sum() / max(len(df), 1) > 0.8:
                 df[col] = coerced

        return df

    # ================================================================== #
    # 2. STRUCTURAL ANALYSIS & CLEANING
    # ================================================================== #

    def data_summary(self) -> None:
        """
        Print a structured summary of the dataset.

        Outputs
        -------
        - Shape (rows × columns)
        - Preview of first 20 rows
        - Numeric vs categorical column breakdown
        - Per-column missing value counts and percentages
        - Basic descriptive statistics for numeric columns
        """
        self._check_loaded()
        df = self.df

        print("=" * 70)
        print(f"  DATASET SUMMARY  —  {df.shape[0]:,} rows × {df.shape[1]} columns")
        print("=" * 70)

        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
        print(f"\n  Numeric  columns ({len(num_cols)}): {num_cols}")
        print(f"  Categorical columns ({len(cat_cols)}): {cat_cols}")

        # Missing values
        missing = df.isnull().sum()
        missing_pct = (missing / len(df) * 100).round(2)
        miss_df = pd.DataFrame({"Missing": missing, "Pct %": missing_pct})
        miss_df = miss_df[miss_df["Missing"] > 0].sort_values("Pct %", ascending=False)
        if not miss_df.empty:
            print("\n  Missing Values:")
            print(miss_df.to_string())
        else:
            print("\n  ✅  No missing values found.")

        print("\n  Preview (first 20 rows):")
        try:
            from IPython.display import display  # type: ignore
            display(df.head(20))
        except ImportError:
            print(df.head(20).to_string())

        print("\n  Descriptive Statistics (numeric):")
        try:
            from IPython.display import display
            display(df[num_cols].describe().T)
        except ImportError:
            print(df[num_cols].describe().T.to_string())

    # ------------------------------------------------------------------ #
    def handle_missing_values(
        self,
        strategy: Literal["mean", "median", "mode", "constant"] = "median",
        constant_value: Union[int, float, str, None] = 0,
        columns: Optional[List[str]] = None,
    ) -> None:
        """
        Impute missing values across the dataset.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant'}
            Imputation strategy.
            - *mean* / *median* apply only to numeric columns.
            - *mode* applies to all columns.
            - *constant* fills with ``constant_value``.
        constant_value : scalar, optional
            Used when ``strategy='constant'``.
        columns : list of str, optional
            Restrict imputation to these columns.  Defaults to all columns.
        """
        self._check_loaded()
        cols = columns or self.df.columns.tolist()
        num_cols = self.df[cols].select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df[cols].select_dtypes(exclude=np.number).columns.tolist()

        if strategy == "mean":
            for c in num_cols:
                self.df[c].fillna(self.df[c].mean(), inplace=True)
        elif strategy == "median":
            for c in num_cols:
                self.df[c].fillna(self.df[c].median(), inplace=True)
        elif strategy == "mode":
            for c in cols:
                self.df[c].fillna(self.df[c].mode().iloc[0], inplace=True)
        elif strategy == "constant":
            for c in cols:
                self.df[c].fillna(constant_value, inplace=True)
        else:
            raise ValueError(f"Unknown strategy '{strategy}'. Choose: mean, median, mode, constant.")

        remaining = self.df[cols].isnull().sum().sum()
        print(f"✅  Imputation ({strategy}) complete. Remaining nulls in scope: {remaining}")

    # ------------------------------------------------------------------ #
    def remove_duplicates(self) -> None:
        """
        Drop exact duplicate rows from the dataset in-place.

        Prints the number of duplicates removed.
        """
        self._check_loaded()
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)
        print(f"✅  Removed {removed:,} duplicate row(s). Remaining: {len(self.df):,}")

    # ------------------------------------------------------------------ #
    def handle_outliers(
        self,
        columns: Optional[List[str]] = None,
        action: Literal["flag", "delete"] = "flag",
        iqr_factor: float = 1.5,
    ) -> None:
        """
        Detect and optionally remove IQR-based outliers.

        Parameters
        ----------
        columns : list of str, optional
            Numeric columns to inspect.  Defaults to all numeric columns.
        action : {'flag', 'delete'}
            - *flag*  – add boolean columns ``<col>_outlier`` to the DataFrame.
            - *delete* – drop rows where **any** of the specified columns is an outlier.
        iqr_factor : float
            Multiplier for IQR fence (default 1.5; use 3.0 for extreme outliers).
        """
        self._check_loaded()
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cols = columns or num_cols
        cols = [c for c in cols if c in num_cols]

        outlier_mask = pd.Series(False, index=self.df.index)
        for c in cols:
            q1 = self.df[c].quantile(0.25)
            q3 = self.df[c].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - iqr_factor * iqr, q3 + iqr_factor * iqr
            mask = (self.df[c] < lower) | (self.df[c] > upper)
            n_out = mask.sum()
            print(f"  {c}: {n_out} outlier(s)  [fence: {lower:.2f} – {upper:.2f}]")
            if action == "flag":
                self.df[f"{c}_outlier"] = mask
            outlier_mask |= mask

        if action == "delete":
            before = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"✅  Deleted {before - len(self.df):,} outlier row(s). Remaining: {len(self.df):,}")
        else:
            print(f"✅  Flagged outliers.  New columns added with '_outlier' suffix.")

    # ------------------------------------------------------------------ #
    def delete_rows(self, indices: Optional[str] = None) -> None:
        """
        Interactively delete rows by index.

        Parameters
        ----------
        indices : str, optional
            Comma-separated row indices to drop (e.g. ``"0, 5, 12"``).
            If *None*, the user is prompted for input.
        """
        self._check_loaded()
        if indices is None:
            indices = input("Enter comma-separated row indices to delete: ")
        idx = [int(i.strip()) for i in indices.split(",") if i.strip().isdigit()]
        valid = [i for i in idx if i in self.df.index]
        self.df.drop(index=valid, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f"✅  Deleted rows: {valid}. Remaining: {len(self.df):,}")

    # ------------------------------------------------------------------ #
    def delete_columns(self, columns: Optional[str] = None) -> None:
        """
        Interactively delete columns by name.

        Parameters
        ----------
        columns : str, optional
            Comma-separated column names to drop (e.g. ``"Name, Ticket"``).
            If *None*, the user is prompted for input.
        """
        self._check_loaded()
        if columns is None:
            columns = input("Enter comma-separated column names to delete: ")
        cols = [c.strip() for c in columns.split(",")]
        valid = [c for c in cols if c in self.df.columns]
        self.df.drop(columns=valid, inplace=True)
        print(f"✅  Deleted columns: {valid}. Remaining columns: {list(self.df.columns)}")

    # ================================================================== #
    # 3. FEATURE ENGINEERING PREPARATION
    # ================================================================== #

    def extract_normalized_numeric_data(
        self,
        method: Literal["minmax", "standard", "robust"] = "standard",
        columns: Optional[List[str]] = None,
    ) -> pd.DataFrame:
        """
        Scale numeric columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : {'minmax', 'standard', 'robust'}
            - *minmax*   – scales to [0, 1].
            - *standard* – zero mean, unit variance (Z-score).
            - *robust*   – centres on median, scales by IQR (outlier-resistant).
        columns : list of str, optional
            Columns to scale.  Defaults to all numeric columns.

        Returns
        -------
        pd.DataFrame
            Scaled numeric DataFrame.
        """
        self._check_loaded()
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cols = columns or num_cols

        scalers = {"minmax": MinMaxScaler(),
                   "standard": StandardScaler(),
                   "robust": RobustScaler()}
        if method not in scalers:
            raise ValueError(f"Unknown method '{method}'. Choose: minmax, standard, robust.")

        scaler = scalers[method]
        subset = self.df[cols].copy()
        scaled = pd.DataFrame(scaler.fit_transform(subset), columns=cols, index=self.df.index)
        print(f"✅  Numeric scaling ({method}) applied to {len(cols)} columns.")
        return scaled

    # ------------------------------------------------------------------ #
    def extract_normalized_categorical_data(
        self,
        method: Literal["onehot", "ordinal", "uniform"] = "onehot",
        columns: Optional[List[str]] = None,
    ) -> pd.DataFrame:
        """
        Encode categorical columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : {'onehot', 'ordinal', 'uniform'}
            - *onehot*   – one-hot encoding (``pd.get_dummies``).
            - *ordinal*  – integer label encoding (0, 1, 2 …).
            - *uniform*  – ordinal then scaled to [0, 1].
        columns : list of str, optional
            Columns to encode.  Defaults to all object/category columns.

        Returns
        -------
        pd.DataFrame
            Encoded DataFrame.
        """
        self._check_loaded()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()
        cols = columns or cat_cols

        if method == "onehot":
            encoded = pd.get_dummies(
                self.df[cols].fillna("Missing"),
                drop_first=False
            )
            encoded = encoded.astype(int)
        elif method in ("ordinal", "uniform"):
            encoded = pd.DataFrame(index=self.df.index)
            le = LabelEncoder()
            for c in cols:
                encoded[c] = le.fit_transform(self.df[c].astype(str))
            if method == "uniform":
                encoded = encoded.apply(lambda s: (s - s.min()) / (s.max() - s.min())
                                        if s.max() != s.min() else s * 0.0)
        else:
            raise ValueError(f"Unknown method '{method}'. Choose: onehot, ordinal, uniform.")

        print(f"✅  Categorical encoding ({method}) applied to {len(cols)} columns → "
              f"{encoded.shape[1]} output columns.")
        return encoded

    # ------------------------------------------------------------------ #
    def get_merged_dataset(
        self,
        numeric_method: Literal["minmax", "standard", "robust"] = "standard",
        categorical_method: Literal["onehot", "ordinal", "uniform"] = "ordinal",
    ) -> pd.DataFrame:
        """
        Return a unified DataFrame of scaled numeric + encoded categorical data.

        Parameters
        ----------
        numeric_method : str
            Passed to :meth:`extract_normalized_numeric_data`.
        categorical_method : str
            Passed to :meth:`extract_normalized_categorical_data`.

        Returns
        -------
        pd.DataFrame
        """
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=categorical_method)
        merged = pd.concat([num_df, cat_df], axis=1)
        print(f"✅  Merged dataset shape: {merged.shape}")
        return merged

    # ================================================================== #
    # 4. ADVANCED INTERACTIVE VISUALISATION
    # ================================================================== #

    def plot_univariate(self, column: str) -> None:
        """
        Render a 3-panel subplot for a single numeric column.

        Panels
        ------
        1. Horizontal Violin + Box (left)
        2. Scatter plot – index vs value (middle)
        3. Histogram with KDE overlay (right)

        Parameters
        ----------
        column : str
            Name of a numeric column.
        """
        self._check_loaded()
        if column not in self.df.columns:
            print(f"Column '{column}' not found.")
            return
        series = self.df[column].dropna()
        if series.empty:
            print(f"Column '{column}' is entirely null after dropping NaN.")
            return

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=["Violin + Box", "Index vs Value", "Histogram + KDE"],
        )

        # Panel 1 – Violin + Box
        fig.add_trace(go.Violin(x=series, name=column, box_visible=True,
                                meanline_visible=True, orientation="h",
                                fillcolor="#636EFA", opacity=0.6,
                                line_color="darkblue"), row=1, col=1)

        # Panel 2 – Scatter
        fig.add_trace(go.Scatter(x=series.index, y=series.values,
                                 mode="markers",
                                 marker=dict(color="#EF553B", size=4, opacity=0.6),
                                 name=column), row=1, col=2)

        # Panel 3 – Histogram + KDE
        fig.add_trace(go.Histogram(x=series, nbinsx=30, name="Count",
                                   marker_color="#00CC96", opacity=0.7), row=1, col=3)
        kde_x = np.linspace(series.min(), series.max(), 300)
        kde = stats.gaussian_kde(series)
        scale = len(series) * (series.max() - series.min()) / 30
        fig.add_trace(go.Scatter(x=kde_x, y=kde(kde_x) * scale,
                                 mode="lines", name="KDE",
                                 line=dict(color="black", width=2)), row=1, col=3)

        fig.update_layout(
            title_text=f"Univariate Analysis — {column}",
            template="plotly_white",
            height=420,
            showlegend=False,
        )
        fig.show()

    # ------------------------------------------------------------------ #
    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """
        Plot the relationship between two columns using type-aware chart selection.

        Rules
        -----
        - **Num–Num**  → scatter plot with OLS trendline.
        - **Cat–Num**  → box plot with individual data points overlaid.
        - **Cat–Cat**  → grouped (side-by-side) bar chart.

        Parameters
        ----------
        col_x : str
            First column name.
        col_y : str
            Second column name.
        """
        self._check_loaded()
        for c in (col_x, col_y):
            if c not in self.df.columns:
                print(f"Column '{c}' not found.")
                return

        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        if x_num and y_num:
            fig = px.scatter(self.df, x=col_x, y=col_y,
                             trendline="ols",
                             title=f"Scatter: {col_x} vs {col_y}",
                             template="plotly_white",
                             opacity=0.6)
        elif not x_num and y_num:
            fig = px.box(self.df, x=col_x, y=col_y,
                         points="all",
                         title=f"Box: {col_y} by {col_x}",
                         template="plotly_white")
        elif x_num and not y_num:
            fig = px.box(self.df, x=col_y, y=col_x,
                         points="all",
                         title=f"Box: {col_x} by {col_y}",
                         template="plotly_white")
        else:
            # Cat-Cat
            counts = (self.df.groupby([col_x, col_y])
                      .size().reset_index(name="Count"))
            fig = px.bar(counts, x=col_x, y="Count", color=col_y,
                         barmode="group",
                         title=f"Grouped Bar: {col_x} × {col_y}",
                         template="plotly_white")

        fig.show()

    # ------------------------------------------------------------------ #
    def plot_categorical_frequency(self, column: str) -> None:
        """
        Bar chart of value counts with raw counts and percentage labels.

        Parameters
        ----------
        column : str
            Name of a categorical column.
        """
        self._check_loaded()
        if column not in self.df.columns:
            print(f"Column '{column}' not found.")
            return

        counts = self.df[column].value_counts()
        pct = (counts / counts.sum() * 100).round(1)
        labels = [f"{v} ({p}%)" for v, p in zip(counts.values, pct)]

        fig = go.Figure(go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            text=labels,
            textposition="outside",
            marker_color=px.colors.qualitative.Plotly[:len(counts)],
        ))
        fig.update_layout(
            title=f"Frequency: {column}",
            xaxis_title=column,
            yaxis_title="Count",
            template="plotly_white",
        )
        fig.show()

    # ================================================================== #
    # 5. DEEP STATISTICAL INSIGHTS
    # ================================================================== #

    def plot_all_associations_heatmap(self) -> None:
        """
        Plot a unified association heatmap across all column types.

        Metric selection
        ----------------
        - **Numeric–Numeric**    : Pearson's *r*
        - **Categorical–Categorical** : Cramér's *V*
        - **Mixed (Num–Cat)**    : Eta (η) squared via one-way ANOVA
          (equivalent to Point-Biserial *r* for binary categories)

        The resulting *n × n* matrix is displayed as an annotated Plotly heatmap.
        """
        self._check_loaded()
        df = self.df.copy()

        # Drop flag columns injected by handle_outliers
        cols = [c for c in df.columns if not c.endswith("_outlier")]
        df = df[cols]

        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
        all_cols = num_cols + cat_cols
        n = len(all_cols)

        matrix = np.full((n, n), np.nan)
        for i, ci in enumerate(all_cols):
            for j, cj in enumerate(all_cols):
                si = df[ci].dropna()
                sj = df[cj].dropna()
                common = df[[ci, cj]].dropna()
                if common.empty:
                    continue
                ci_num = ci in num_cols
                cj_num = cj in num_cols

                if ci_num and cj_num:
                    try:
                        r, _ = stats.pearsonr(common[ci], common[cj])
                        matrix[i, j] = r
                    except Exception:
                        matrix[i, j] = 0
                elif not ci_num and not cj_num:
                    matrix[i, j] = _cramers_v(common[ci].astype(str),
                                              common[cj].astype(str))
                else:
                    num_col = ci if ci_num else cj
                    cat_col = cj if ci_num else ci
                    matrix[i, j] = _eta_squared(common[num_col],
                                                common[cat_col].astype(str))

        text = np.where(np.isnan(matrix), "", np.round(matrix, 2).astype(str))
        fig = go.Figure(go.Heatmap(
            z=matrix,
            x=all_cols,
            y=all_cols,
            text=text,
            texttemplate="%{text}",
            colorscale="RdBu",
            zmid=0,
            zmin=-1,
            zmax=1,
            colorbar=dict(title="Association"),
        ))
        fig.update_layout(
            title=("All-pairs Association Heatmap<br>"
                   "<sup>Num–Num: Pearson r  |  Cat–Cat: Cramér V  |  "
                   "Mixed: Eta²</sup>"),
            template="plotly_white",
            height=max(500, 60 * n),
            width=max(600, 70 * n),
            xaxis_tickangle=-45,
        )
        fig.show()

    # ================================================================== #
    # UTILITIES
    # ================================================================== #

    def reset(self) -> None:
        """Restore the dataset to its original state after upload/load."""
        if self._original_df is None:
            print("No original dataset stored.")
            return
        self.df = self._original_df.copy()
        print("✅  Dataset reset to original state.")

    def get_numeric_columns(self) -> List[str]:
        """Return a list of numeric column names."""
        self._check_loaded()
        return self.df.select_dtypes(include=np.number).columns.tolist()

    def get_categorical_columns(self) -> List[str]:
        """Return a list of categorical (non-numeric) column names."""
        self._check_loaded()
        return self.df.select_dtypes(exclude=np.number).columns.tolist()

    def _check_loaded(self) -> None:
        """Raise RuntimeError if no dataset has been loaded."""
        if self.df is None:
            raise RuntimeError("No dataset loaded. Call upload_data() or load_dataframe() first.")
if __name__ == "__main__":
    di = DataInspector()
    print("DataInspector loaded successfully!")


DataInspector loaded successfully!
